# Preprocessing (2025)

Filter and clean raw Statcast pitches **before** feature engineering.

**Input:** `data/statcast_2025.parquet` (`00_data_pull.ipynb`) — must be **schema v3** (includes `release_pos_x/y/z`).

**Output:** `data/preprocessed_2025.parquet`

Logic lives in `src/preprocessing.py`. Run **`03_feature_engineering.ipynb`** next.

> **If you see missing `release_pos_*` columns:** your pull cache is stale. Run `00_data_pull.ipynb` with `FORCE_PULL = True`, or:
> `.\.venv\Scripts\python.exe -m src.statcast_pull --force`

## Filters (applied in order)

| Step | Rule |
|------|------|
| Regular season | `game_type == 'R'` when present, dates **2025-03-27 → 2025-09-28** |
| Competitive pitches | FF, SI, FC, CU, SV, KC, SL, ST, CH, FS |
| Batters | **All batters kept** — the model must score full lineups, including part-time players |
| Velocity outliers | Drop pitches **below** Tukey lower fence on `release_speed` (**1.5× IQR**, per pitch type). High velocities kept. |
| Base state | Keep only the eight defined runner configurations |
| Required fields | Drop rows missing location, zone height, count, movement, velocity, or `p_throws` |

Runner columns (`on_1b` / `on_2b` / `on_3b`) use **NA = empty base** — not treated as missing data.

**Qualified hitters (502+ AB)** are **not** filtered here. That subset is for the app / leaderboard only — see `src/player_lookup.py` (`filter_qualified_batters`).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.statcast_schema import REQUIRED_STATCAST_COLS, SCHEMA_VERSION, STATCAST_KEEP_COLS

import importlib
import src.preprocessing as preprocessing

importlib.reload(preprocessing)
build_preprocessed_frame = preprocessing.build_preprocessed_frame

DATA_FILE = ROOT / "data" / "statcast_2025.parquet"
OUTPUT_FILE = ROOT / "data" / "preprocessed_2025.parquet"

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Run 00_data_pull.ipynb first — missing {DATA_FILE}")

raw = pd.read_parquet(DATA_FILE)
missing = sorted(REQUIRED_STATCAST_COLS - set(raw.columns))
if missing:
    raise ValueError(
        f"statcast_2025.parquet is missing {missing}. "
        f"Expected schema v{SCHEMA_VERSION} ({len(STATCAST_KEEP_COLS)} columns). "
        "Re-run the pull with FORCE_PULL=True in 00_data_pull.ipynb, or:\n"
        "  .\\.venv\\Scripts\\python.exe -m src.statcast_pull --force"
    )

preprocessed, report = build_preprocessed_frame(raw)

print("Preprocessing summary")
print(f"• Raw pitches: {report.n_raw:,}")
print(f"• After regular-season filter: {report.n_regular_season:,}")
print(f"• After competitive pitch types: {report.n_competitive:,}")
print(f"• Unique batters: {report.n_unique_batters:,}")
print(f"• Low-velocity outliers removed: {report.n_velocity_outliers_removed:,}")
print(f"• Invalid base states dropped: {report.n_invalid_base_state:,}")
print(f"• Missing required fields dropped: {report.n_missing_required:,}")
print(f"• Final preprocessed frame: {report.n_final:,}")

print("\nVelocity outlier removal (lower fence only, per pitch type)")
display(report.velocity_outlier_summary)

preprocessed.to_parquet(OUTPUT_FILE, index=False)
print(f"\nSaved → {OUTPUT_FILE}")

Preprocessing summary
• Raw pitches: 711,897
• After regular-season filter: 711,897
• After competitive pitch types: 706,051
• Unique batters: 673
• Low-velocity outliers removed: 8,539
• Invalid base states dropped: 0
• Missing required fields dropped: 1,890
• Final preprocessed frame: 695,622

Velocity outlier removal (lower fence only, per pitch type)


,pitch_type,n_pitches,q1_mph,q3_mph,lower_fence_mph,removed,min_removed_mph
0,CH,73307,83.8,88.4,76.90,812,49.3
1,CU,47528,77.6,82.0,71.00,500,34.2
2,FC,53443,87.7,91.5,82.00,301,76.2
3,FF,226025,92.9,96.2,87.95,1970,77.7
4,FS,23084,83.9,88.5,77.00,1,76.2
5,KC,12648,79.7,85.7,70.70,20,67.7
6,SI,110141,92.1,95.7,86.70,1947,81.0
7,SL,102078,84.6,88.2,79.20,2373,32.8
8,ST,54232,80.8,84.6,75.10,780,67.7
9,SV,3562,80.2,83.4,75.40,23,72.4



Saved → C:\Users\tabshire\Desktop\Portfolio\data\preprocessed_2025.parquet


In [2]:
print(f"Columns saved: {len(preprocessed.columns)}")
display(preprocessed.head())

Columns saved: 35


,game_date,game_pk,at_bat_number,pitch_number,batter,pitcher,pitch_type,pitch_name,p_throws,stand,...,pfx_x,pfx_z,release_spin_rate,spin_axis,release_extension,release_pos_x,release_pos_y,release_pos_z,bat_speed,attack_angle
0,2025-03-27,778545,1,1,595777,650633,FF,4-Seam Fastball,R,L,...,-0.88,1.75,2333.0,215.0,5.9,-3.10,54.599998,5.81,NaN,NaN
1,2025-03-27,778545,1,2,595777,650633,SI,Sinker,R,L,...,-1.64,0.18,2334.0,229.0,5.7,-3.27,54.759998,5.45,NaN,NaN
2,2025-03-27,778545,1,3,595777,650633,CH,Changeup,R,L,...,-1.93,0.58,2042.0,245.0,5.9,-3.25,54.590000,5.40,78.599998,10.611278
3,2025-03-27,778545,2,1,663586,650633,FF,4-Seam Fastball,R,R,...,-1.05,1.52,2470.0,215.0,6.1,-3.02,54.400002,5.71,79.500000,2.776964
4,2025-03-27,778545,2,2,663586,650633,FF,4-Seam Fastball,R,R,...,-0.72,1.52,2493.0,213.0,6.1,-2.88,54.450001,5.77,NaN,NaN
